In [2]:
import google.generativeai
!pip install pypdf
import json
import os
import pandas
!pip install openai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 5.4 MB/s eta 0:00:00


Using Gemini API

In [ ]:
import os
#os.environ["GEMINI_API_KEY"] = "APIKEY" 
os.environ["GEMINI_API_KEY"] = "APIKEY"  #Phase2
#os.environ["GEMINI_API_KEY"] = "APIKEY"  


In [ ]:
import google.generativeai as genai

genai.configure(api_key="APIKEY")

# List available models to find the correct one
for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

model = genai.GenerativeModel("gemini-2.5-flash")

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

Using OpenAI for API due to Gemini rate limit

In [ ]:
from openai import OpenAI
client = OpenAI(api_key="APIKEY")

def analyze_document(text, prompt):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt + "\n\nDocument:\n" + text[:12000]}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [6]:
from google.colab import files

uploaded = files.upload()

Saving 1_Training Activities Manual.pdf to 1_Training Activities Manual.pdf
Saving 2_VisiOn Pilots Reports.pdf to 2_VisiOn Pilots Reports.pdf
Saving 3_VisiOn Pilots Preparation.pdf to 3_VisiOn Pilots Preparation.pdf
Saving 4_Scenarios for the Pilots.pdf to 4_Scenarios for the Pilots.pdf
Saving 5_VisiOn Privacy Platform Testing Results.pdf to 5_VisiOn Privacy Platform Testing Results.pdf
Saving 6_Dissemination and Standardisation Activities Report.pdf to 6_Dissemination and Standardisation Activities Report.pdf
Saving 7_Citizens and Public Administrations Privacy Requirements.pdf to 7_Citizens and Public Administrations Privacy Requirements.pdf


In [7]:
from pypdf import PdfReader

def load_pdf_text(pdf_file):
    reader = PdfReader(pdf_file)
    text = ""

    for page in reader.pages:
        if page.extract_text():
            text += page.extract_text()

    return text

Analysis for CRA Law

In [55]:
cra_prompt = """
You are a legal and technical compliance analyst specializing in EU cybersecurity law and software quality engineering.

Analyze the following project deliverable in relation to the Cyber Resilience Act (CRA, Regulation (EU) 2024/2847, Annex I, Section 1.3 a-k).

Focus on:
- ISO/IEC 25010 quality characteristics: Security (Confidentiality, Integrity, Authenticity), Reliability, and Maintainability.
- SDLC phases: Development, Testing, Deployment, Maintenance.
- Artifacts: source code, architecture documentation, security requirements, test reports, deployment configurations.
- Activities: secure implementation, security testing, vulnerability management, patch and update processes.

Decision Rule for Applies_IF:
- Return "YES" if there is a 70% or higher likelihood that the deliverable describes a hardware or software product with a direct or indirect logical or physical data connection to a device or network placed or intended for the EU market (Art. 3(1)). Partial evidence such as mentions of platforms, web services, APIs, user authentication, data transmission, or network-connected components is sufficient to reach the 70% threshold.
- Return "NO" only if there is clear evidence that the product has no network connectivity or is explicitly outside the EU market scope.
- Do NOT return UNCLEAR. Make a definitive decision based on all available evidence and reasonable inference.

Return your answer as structured JSON:
{
  "Applies_IF": "YES | NO",
  "Justification": "Explain your reasoning and confidence level (e.g., 80% likely)
                    based on evidence found in the deliverable text.",
  "Applies_WHERE": ["List specific components, modules, or systems where CRA
                    Annex I requirements are triggered."],
  "Implementation_HOW": [
    "Minimise attack surface (Annex I, 1.3(a))",
    "Implement secure default configuration (Annex I, 1.3(c))",
    "Protect data confidentiality and integrity (Annex I, 1.3(d,e))",
    "Provide SBOM and free security updates (Annex I, 2.1)",
    "Report exploited vulnerabilities to ENISA within 24h (Art. 14)"
  ]
}

Deliverable text:
{deliverable_text}
"""

In [56]:
from pypdf import PdfReader
from io import BytesIO

def load_pdf_text(pdf_bytes):
    reader = PdfReader(BytesIO(pdf_bytes))
    text = ""

    for page in reader.pages:
        if page.extract_text():
            text += page.extract_text()

    return text

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key="APIKEY")

def analyze_document(text, prompt):

    full_prompt = prompt + "\n\nDocument:\n" + text[:12000]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": full_prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}   #  forces JSON
    )

    return json.loads(response.choices[0].message.content)

In [58]:
import pandas as pd

results = []

for pdf_file, pdf_bytes in uploaded.items():

    print("Processing:", pdf_file)

    text = load_pdf_text(pdf_bytes)

    analysis = analyze_document(text, cra_prompt)

    results.append({
        "document": pdf_file,
        "law": "CRA",
        "analysis": analysis,
        "Applies_IF": analysis.get("Applies_IF"),
        "Justification": analysis.get("Justification"),
        "Applies_WHERE": analysis.get("Applies_WHERE"),
        "Implementation_HOW": analysis.get("Implementation_HOW")
    })

df = pd.DataFrame(results)
df

Processing: 1_Training Activities Manual.pdf
Processing: 2_VisiOn Pilots Reports.pdf
Processing: 3_VisiOn Pilots Preparation.pdf
Processing: 4_Scenarios for the Pilots.pdf
Processing: 5_VisiOn Privacy Platform Testing Results.pdf
Processing: 6_Dissemination and Standardisation Activities Report.pdf
Processing: 7_Citizens and Public Administrations Privacy Requirements.pdf


,document,law,analysis,Applies_IF,Justification,Applies_WHERE,Implementation_HOW
0,1_Training Activities Manual.pdf,CRA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes various tools and fr...,"[Dynamic Audit Engine (DAE), Socio-Technical S...","[Minimise attack surface (Annex I, 1.3(a)), Im..."
1,2_VisiOn Pilots Reports.pdf,CRA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable is part of a project funded by...,"[User authentication mechanisms, Data transmis...","[Minimise attack surface (Annex I, 1.3(a)), Im..."
2,3_VisiOn Pilots Preparation.pdf,CRA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes a project that invol...,"[VisiOn Privacy Platform, Pilot Sites (Ministe...","[Minimise attack surface (Annex I, 1.3(a)), Im..."
3,4_Scenarios for the Pilots.pdf,CRA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes a project that invol...,"[VisiOn Privacy Platform (VPP), Data Managemen...","[Minimise attack surface (Annex I, 1.3(a)), Im..."
4,5_VisiOn Privacy Platform Testing Results.pdf,CRA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes a project that invol...,"[Citizen authentication server, Web Framework,...","[Minimise attack surface (Annex I, 1.3(a)), Im..."
5,6_Dissemination and Standardisation Activities...,CRA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes a project that is pa...,"[VisiOn project website, Online dissemination ...","[Minimise attack surface (Annex I, 1.3(a)), Im..."
6,7_Citizens and Public Administrations Privacy ...,CRA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable discusses user requirements fo...,[User requirements related to data privacy and...,"[Minimise attack surface (Annex I, 1.3(a)), Im..."


In [59]:
df.to_csv("openAICRA_law_analysis_results.csv", index=False)
files.download("openAICRA_law_analysis_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Analysis for CSA Law

In [32]:
csa_prompt = """
You are a legal and technical compliance analyst specializing in EU cybersecurity law.

Analyze the following project deliverable in relation to the EU Cybersecurity Act (CSA, Regulation (EU) 2019/881).

Focus on:
- ISO/IEC 25010 quality characteristics: Security (all sub-characteristics), Reliability.
- SDLC phases: Development, Testing, Deployment, Maintenance.
- Whether the artifact describes ICT products or services that are placed on the EU market and could be subject to an ENISA certification scheme.

Decision Rule for Applies_IF:
- Return "YES" if there is a 70% or higher likelihood that the deliverable describes ICT products, services, or processes that require or benefit from compliance with specified cybersecurity requirements under an EU certification scheme (Art. 49). Partial evidence such as mentions of software tools, digital platforms, ICT services, citizen-facing applications, or security-relevant components is sufficient to reach the 70% threshold.
- Return "NO" only if there is clear evidence that no ICT products or services with potential certification needs are described.
- Do NOT return UNCLEAR. Make a definitive decision based on all available evidence and reasonable inference.

Return your answer as structured JSON:
{
  "Applies_IF": "YES | NO",
  "Justification": "Explain your reasoning and confidence level (e.g., 75% likely)
                    based on evidence found in the deliverable text.",
  "Applies_WHERE": ["List specific ICT tools, platforms, or services in scope."],
  "Implementation_HOW": {
    "Documentation": "Publish user guidance for secure installation and
                      maintenance (Art. 55(1)).",
    "Security_Support": "Declare duration of security support and reference
                         public vulnerability repositories (Art. 55(3)).",
    "Certification": "Evaluate applicability of ENISA certification schemes
                      at basic, substantial, or high assurance level."
  }
}

Deliverable text:
{deliverable_text}
"""

In [33]:
from pypdf import PdfReader
from io import BytesIO

def load_pdf_text(pdf_bytes):
    reader = PdfReader(BytesIO(pdf_bytes))
    text = ""

    for page in reader.pages:
        if page.extract_text():
            text += page.extract_text()

    return text

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key="APIKEY")

def analyze_document(text, prompt):

    full_prompt = prompt + "\n\nDocument:\n" + text[:12000]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": full_prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}   #  forces JSON
    )

    return json.loads(response.choices[0].message.content)

In [36]:
import pandas as pd

results = []

for pdf_file, pdf_bytes in uploaded.items():

    print("Processing:", pdf_file)

    text = load_pdf_text(pdf_bytes)

    analysis = analyze_document(text, csa_prompt)

    results.append({
        "document": pdf_file,
        "law": "CSA",
        "analysis": analysis,
        "Applies_IF": analysis.get("Applies_IF"),
        "Justification": analysis.get("Justification"),
        "Applies_WHERE": analysis.get("Applies_WHERE"),
        "Implementation_HOW": analysis.get("Implementation_HOW")
    })

df = pd.DataFrame(results)
df

Processing: 1_Training Activities Manual.pdf
Processing: 2_VisiOn Pilots Reports.pdf
Processing: 3_VisiOn Pilots Preparation.pdf
Processing: 4_Scenarios for the Pilots.pdf
Processing: 5_VisiOn Privacy Platform Testing Results.pdf
Processing: 6_Dissemination and Standardisation Activities Report.pdf
Processing: 7_Citizens and Public Administrations Privacy Requirements.pdf


,document,law,analysis,Applies_IF,Justification,Applies_WHERE,Implementation_HOW
0,1_Training Activities Manual.pdf,CSA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes various ICT tools an...,"[Dynamic Audit Engine (DAE), Socio-Technical S...",{'Documentation': 'Publish user guidance for s...
1,2_VisiOn Pilots Reports.pdf,CSA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes a project focused on...,"[Privacy management tools, User-centric applic...",{'Documentation': 'Publish user guidance for s...
2,3_VisiOn Pilots Preparation.pdf,CSA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,"The deliverable describes the VisiOn project, ...","[VisiOn Privacy Platform, Pilot Sites' ICT sys...",{'Documentation': 'Publish user guidance for s...
3,4_Scenarios for the Pilots.pdf,CSA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes scenarios for pilot ...,"[VisiOn Privacy Platform (VPP), STS-tool (Soci...",{'Documentation': 'Publish user guidance for s...
4,5_VisiOn Privacy Platform Testing Results.pdf,CSA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes the VisiOn Privacy P...,"[VisiOn Privacy Platform, Citizen authenticati...",{'Documentation': 'Publish user guidance for s...
5,6_Dissemination and Standardisation Activities...,CSA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes dissemination and st...,"[VisiOn project website, Publications related ...",{'Documentation': 'Publish user guidance for s...
6,7_Citizens and Public Administrations Privacy ...,CSA,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable describes user requirements fo...,"[VisiOn platform, Public administration applic...",{'Documentation': 'Publish user guidance for s...


In [37]:
df.to_csv("openAI_CSA_law_analysis_results.csv", index=False)
files.download("openAI_CSA_law_analysis_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Analysis for NIS2 Law

In [42]:
nis2_prompt = """
You are a legal and technical compliance analyst specializing in EU cybersecurity law.

Analyze the following project deliverable in relation to the NIS2 Directive (Directive (EU) 2022/2555).

Focus on:
- ISO/IEC 25010 quality characteristics: Security, Reliability (availability, fault tolerance), Maintainability.
- SDLC phases: Development (secure acquisition, Art. 21(2)(e)), Deployment (access control, Art. 21(2)(i)), Maintenance (incident handling, Art. 21(2)(b)).
- Whether the entity responsible for this deliverable operates in a critical sector (NIS2 Annexes I or II) and qualifies as medium-sized or larger.

Decision Rule for Applies_IF:
- Return "YES" if there is a 70% or higher likelihood that the entity responsible for this deliverable qualifies as an essential or important entity under NIS2 Annexes I or II (Art. 3).
  Partial evidence such as mentions of hospitals, government bodies, public administrations, energy providers, digital infrastructure, or any Annex I/II sector keywords is sufficient to reach the 70% threshold. If the deliverable involves multiple stakeholders and at least one is likely a critical-sector entity, return "YES".
- Return "NO" only if there is clear evidence that all involved entities are small enterprises or operate exclusively in non-critical sectors.
- Do NOT return UNCLEAR. Make a definitive decision based on all available evidence and reasonable inference.

Return your answer as structured JSON:
{
  "Applies_IF": "YES | NO",
  "Justification": "Explain your reasoning and confidence level (e.g., 70% likely) based on evidence found in the deliverable text.",
  "Applies_WHERE": ["List specific systems, processes, or components where NIS2 Art. 21 measures are triggered."],
  "Implementation_HOW": [
    "Establish management-approved cybersecurity risk management framework (Art. 21(1))",
    "Implement incident reporting: 24h early warning, 72h notification, 1-month report (Art. 23)",
    "Enforce cyber hygiene training programs for all personnel (Art. 21(2)(g))",
    "Adopt all-hazards approach: business continuity, supply chain security (Art. 21(2)(c,d))"
  ]
}

Deliverable text:
{deliverable_text}
"""

In [43]:
from pypdf import PdfReader
from io import BytesIO

def load_pdf_text(pdf_bytes):
    reader = PdfReader(BytesIO(pdf_bytes))
    text = ""

    for page in reader.pages:
        if page.extract_text():
            text += page.extract_text()

    return text

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key="APIKEY")

def analyze_document(text, prompt):

    full_prompt = prompt + "\n\nDocument:\n" + text[:12000]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": full_prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}   #  forces JSON
    )

    return json.loads(response.choices[0].message.content)

In [45]:
import pandas as pd

results = []

for pdf_file, pdf_bytes in uploaded.items():

    print("Processing:", pdf_file)

    text = load_pdf_text(pdf_bytes)

    analysis = analyze_document(text, nis2_prompt)

    results.append({
        "document": pdf_file,
        "law": "NIS2",
        "analysis": analysis,
        "Applies_IF": analysis.get("Applies_IF"),
        "Justification": analysis.get("Justification"),
        "Applies_WHERE": analysis.get("Applies_WHERE"),
        "Implementation_HOW": analysis.get("Implementation_HOW")
    })

df = pd.DataFrame(results)
df

Processing: 1_Training Activities Manual.pdf
Processing: 2_VisiOn Pilots Reports.pdf
Processing: 3_VisiOn Pilots Preparation.pdf
Processing: 4_Scenarios for the Pilots.pdf
Processing: 5_VisiOn Privacy Platform Testing Results.pdf
Processing: 6_Dissemination and Standardisation Activities Report.pdf
Processing: 7_Citizens and Public Administrations Privacy Requirements.pdf


,document,law,analysis,Applies_IF,Justification,Applies_WHERE,Implementation_HOW
0,1_Training Activities Manual.pdf,NIS2,"{'Applies_IF': 'YES', 'Justification': 'The pr...",YES,The project involves training activities and t...,[Training activities related to cybersecurity ...,[Establish management-approved cybersecurity r...
1,2_VisiOn Pilots Reports.pdf,NIS2,"{'Applies_IF': 'YES', 'Justification': 'The pr...",YES,"The project involves multiple stakeholders, in...","[Pilot Implementation processes, Incident hand...",[Establish management-approved cybersecurity r...
2,3_VisiOn Pilots Preparation.pdf,NIS2,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable involves entities such as 'Osp...,[Development phase: Secure acquisition process...,[Establish management-approved cybersecurity r...
3,4_Scenarios for the Pilots.pdf,NIS2,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable involves multiple stakeholders...,[Development phase: Secure acquisition process...,[Establish management-approved cybersecurity r...
4,5_VisiOn Privacy Platform Testing Results.pdf,NIS2,"{'Applies_IF': 'YES', 'Justification': 'The pr...",YES,"The project involves multiple stakeholders, in...","[Testing Phase 1: Connectivity Test Cases, Tes...",[Establish management-approved cybersecurity r...
5,6_Dissemination and Standardisation Activities...,NIS2,"{'Applies_IF': 'YES', 'Justification': 'The pr...",YES,"The project involves multiple stakeholders, in...",[Dissemination activities related to cybersecu...,[Establish management-approved cybersecurity r...
6,7_Citizens and Public Administrations Privacy ...,NIS2,"{'Applies_IF': 'YES', 'Justification': 'The de...",YES,The deliverable involves stakeholders from cri...,[Stakeholder analysis involving Greek public a...,[Establish management-approved cybersecurity r...


In [46]:
df.to_csv("openAI_NIS2_law_analysis_results.csv", index=False)
files.download("openAI_NIS2_law_analysis_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Analysis for DORA Law

In [52]:
dora_prompt = """
You are a legal and technical compliance analyst specializing in EU cybersecurity law.

Analyze the following project deliverable in relation to the Digital Operational Resilience Act (DORA, Regulation (EU) 2022/2554).

Focus on:
- ISO/IEC 25010 quality characteristics: Security, Reliability (availability, fault tolerance), Maintainability.
- SDLC phases: Development (ICT systems acquisition, Art. 8), Testing (resilience testing, Arts. 24-25), Deployment (change management, Art. 9), Maintenance (incident handling, Arts. 17-23).
- Whether the entity is a financial entity or critical ICT third-party provider to financial entities (Art. 2).

Decision Rule for Applies_IF:
- Return "YES" if there is a 70% or higher likelihood that the entity responsible for this deliverable is a financial entity (bank, insurer, investment firm, payment service provider, crypto-asset provider) or a critical ICT third-party provider delivering services to such financial entities within the EU (Art. 2). Partial evidence such as mentions of financial services, payment processing, banking infrastructure, insurance systems, or ICT contracts with financial institutions is sufficient to reach the 70% threshold.
- Return "NO" if the deliverable describes entities or services clearly operating outside the financial sector with no ICT third-party relationship to financial entities.
- Do NOT return UNCLEAR. Make a definitive decision based on all available evidence and reasonable inference.

Return your answer as structured JSON:
{
  "Applies_IF": "YES | NO",
  "Justification": "Explain your reasoning and confidence level (e.g., 95% unlikely — NO) based on evidence found in the deliverable text.",
  "Applies_WHERE": ["List ICT systems, contracts, or processes where DORA obligations are triggered."],
  "Implementation_HOW": [
    "Implement ICT risk management framework (Art. 6)",
    "Classify and report ICT-related incidents (Arts. 17-20)",
    "Conduct digital operational resilience testing (Arts. 24-25)",
    "Manage third-party ICT risks via contractual safeguards (Arts. 28-44)"
  ]
}

Deliverable text:
{deliverable_text}
"""

In [48]:
from pypdf import PdfReader
from io import BytesIO

def load_pdf_text(pdf_bytes):
    reader = PdfReader(BytesIO(pdf_bytes))
    text = ""

    for page in reader.pages:
        if page.extract_text():
            text += page.extract_text()

    return text

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key="APIKEY")

def analyze_document(text, prompt):

    full_prompt = prompt + "\n\nDocument:\n" + text[:12000]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": full_prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}   #  forces JSON
    )

    return json.loads(response.choices[0].message.content)

In [53]:
import pandas as pd

results = []

for pdf_file, pdf_bytes in uploaded.items():

    print("Processing:", pdf_file)

    text = load_pdf_text(pdf_bytes)

    analysis = analyze_document(text, dora_prompt)

    results.append({
        "document": pdf_file,
        "law": "DORA",
        "analysis": analysis,
        "Applies_IF": analysis.get("Applies_IF"),
        "Justification": analysis.get("Justification"),
        "Applies_WHERE": analysis.get("Applies_WHERE"),
        "Implementation_HOW": analysis.get("Implementation_HOW")
    })

df = pd.DataFrame(results)
df

Processing: 1_Training Activities Manual.pdf
Processing: 2_VisiOn Pilots Reports.pdf
Processing: 3_VisiOn Pilots Preparation.pdf
Processing: 4_Scenarios for the Pilots.pdf
Processing: 5_VisiOn Privacy Platform Testing Results.pdf
Processing: 6_Dissemination and Standardisation Activities Report.pdf
Processing: 7_Citizens and Public Administrations Privacy Requirements.pdf


,document,law,analysis,Applies_IF,Justification,Applies_WHERE,Implementation_HOW
0,1_Training Activities Manual.pdf,DORA,"{'Applies_IF': 'NO', 'Justification': 'The del...",NO,The deliverable text does not provide any evid...,[],[]
1,2_VisiOn Pilots Reports.pdf,DORA,"{'Applies_IF': 'NO', 'Justification': 'The del...",NO,The deliverable text does not provide any evid...,[],[]
2,3_VisiOn Pilots Preparation.pdf,DORA,"{'Applies_IF': 'NO', 'Justification': 'The del...",NO,The deliverable text does not provide any evid...,[],[]
3,4_Scenarios for the Pilots.pdf,DORA,"{'Applies_IF': 'NO', 'Justification': 'The del...",NO,The deliverable text does not provide any evid...,[],[]
4,5_VisiOn Privacy Platform Testing Results.pdf,DORA,"{'Applies_IF': 'NO', 'Justification': 'The del...",NO,The deliverable text does not provide any evid...,[],[]
5,6_Dissemination and Standardisation Activities...,DORA,"{'Applies_IF': 'NO', 'Justification': 'The del...",NO,The deliverable text does not provide any evid...,[],[]
6,7_Citizens and Public Administrations Privacy ...,DORA,"{'Applies_IF': 'NO', 'Justification': 'The del...",NO,The deliverable text does not provide any evid...,[],[]


In [54]:
df.to_csv("openAI_DORA_law_analysis_results.csv", index=False)
files.download("openAI_DORA_law_analysis_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>